In [4]:
# Upload the Kaggle ZIP file
from google.colab import files
import zipfile, os

uploaded = files.upload()  # Click "Choose File" and select the downloaded ZIP

# Extract
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
      z.extractall('data')

  # Show what we got
for root, dirs, fnames in os.walk('data'):
      for f in fnames:
          path = os.path.join(root, f)
          print(path, f"({os.path.getsize(path)} bytes)")

Saving archive (3).zip to archive (3).zip
data/xyz_distances.csv (678741 bytes)
data/3d_distances.csv (217611 bytes)
data/angles.csv (100734 bytes)
data/landmarks.csv (1403171 bytes)
data/labels.csv (25126 bytes)


In [ ]:
import pandas as pd
import numpy as np

# Load only the two CSVs we actually need
landmarks = pd.read_csv('data/landmarks.csv')
labels = pd.read_csv('data/labels.csv')
angles = pd.read_csv('data/angles.csv')

print(f"Landmarks: {landmarks.shape[0]} rows, {landmarks.shape[1]} cols")
print(f"Labels:    {labels.shape[0]} rows, {labels.shape[1]} cols")
print(f"Angles:    {angles.shape[0]} rows, {angles.shape[1]} cols")
print(f"\nLandmark columns sample: {list(landmarks.columns[:7])}")
print(f"Angle columns: {list(angles.columns)}")
print(f"\nPose labels: {labels['pose'].value_counts().to_dict()}")

# Merge on pose_id — each row now has landmarks + angles + label
data = landmarks.merge(labels, on='pose_id').merge(angles, on='pose_id')
print(f"\nMerged: {data.shape[0]} rows, {data.shape[1]} cols")
print(f"NaN count: {data.isna().sum().sum()}")

In [ ]:
# Create binary target: 1 = pushup position, 0 = not pushup
pushup_labels = ['pushups_down', 'pushups_up']
data['is_pushup'] = data['pose'].isin(pushup_labels).astype(int)

print(f"Class distribution:")
print(f"  Pushup:     {data['is_pushup'].sum()}")
print(f"  Not pushup: {(data['is_pushup'] == 0).sum()}")
print(f"  Ratio:      1:{(data['is_pushup'] == 0).sum() / data['is_pushup'].sum():.1f}")

In [ ]:
# Map MediaPipe landmark indices to the actual CSV column name prefixes
MP_LANDMARK_NAMES = {
    0: 'nose',
    11: 'left_shoulder',
    12: 'right_shoulder',
    13: 'left_elbow',
    14: 'right_elbow',
    15: 'left_wrist',
    16: 'right_wrist',
    23: 'left_hip',
    24: 'right_hip',
    25: 'left_knee',
    26: 'right_knee',
    27: 'left_ankle',
    28: 'right_ankle',
}
KEY_INDICES = list(MP_LANDMARK_NAMES.keys())  # 13 landmarks

# Build the list of column names we need: x_nose, y_nose, z_nose, x_left_shoulder, ...
landmark_cols = []
for idx in KEY_INDICES:
    name = MP_LANDMARK_NAMES[idx]
    landmark_cols.extend([f'x_{name}', f'y_{name}', f'z_{name}'])

# Verify all columns exist
missing = [c for c in landmark_cols if c not in data.columns]
if missing:
    print(f"ERROR: Missing columns: {missing}")
else:
    print(f"All {len(landmark_cols)} landmark columns found")

# Also grab the 7 angle columns as bonus features
angle_cols = [c for c in data.columns if c in angles.columns and c != 'pose_id']
print(f"Angle columns ({len(angle_cols)}): {angle_cols}")

# Test: show values for first row
print(f"\nFirst row landmarks (nose): {data[['x_nose','y_nose','z_nose']].iloc[0].values}")
print(f"First row label: {data['pose'].iloc[0]}")

In [ ]:
def normalize_landmarks(row):
    """
    Extract 13 key landmarks, center on hip midpoint, scale by torso length.
    Returns normalized 39-value array, or None if torso too small.
    """
    coords = np.array([row[c] for c in landmark_cols], dtype=np.float64).reshape(13, 3)

    # Check for NaN/Inf in raw coords
    if not np.all(np.isfinite(coords)):
        return None

    # Hip midpoint (indices 7=LEFT_HIP, 8=RIGHT_HIP in our 13-key array)
    hip_mid = (coords[7] + coords[8]) / 2

    # Shoulder midpoint (indices 1=LEFT_SHOULDER, 2=RIGHT_SHOULDER)
    shoulder_mid = (coords[1] + coords[2]) / 2

    # Torso length for scale normalization
    torso_len = np.linalg.norm(shoulder_mid - hip_mid)
    if torso_len < 0.01:
        return None

    # Center on hip midpoint and scale by torso length
    normalized = (coords - hip_mid) / torso_len
    return normalized.flatten()

# Process all rows
print("Extracting and normalizing landmarks...")
features = []
labels = []
skipped = 0

for i, row in data.iterrows():
    norm = normalize_landmarks(row)
    if norm is not None:
        # Combine: 39 normalized landmarks + 7 angles (normalized to [0,1])
        angle_vals = np.array([row[c] for c in angle_cols], dtype=np.float64)
        if np.all(np.isfinite(angle_vals)):
            angle_normed = angle_vals / 180.0  # angles are in degrees [0, 180]
            combined = np.concatenate([norm, angle_normed])
            features.append(combined)
            labels.append(row['is_pushup'])
        else:
            skipped += 1
    else:
        skipped += 1

X = np.array(features, dtype=np.float32)
y = np.array(labels, dtype=np.float32)

n_features = 39 + len(angle_cols)
print(f"Feature matrix: {X.shape} (expected Nx{n_features})")
print(f"Labels: {y.shape}, pushup={y.sum():.0f}, not={len(y)-y.sum():.0f}")
print(f"Skipped {skipped} rows (invalid data)")

# Sanity check — this should all be clean now
print(f"\nData quality:")
print(f"  NaN count: {np.isnan(X).sum()}")
print(f"  Inf count: {np.isinf(X).sum()}")
print(f"  Feature range: [{X.min():.3f}, {X.max():.3f}]")

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} samples (pushup={int(y_train.sum())}, not={int(len(y_train)-y_train.sum())})")
print(f"Test:  {X_test.shape[0]} samples (pushup={int(y_test.sum())}, not={int(len(y_test)-y_test.sum())})")

# Use class weights instead of undersampling — keeps all data, penalizes errors on minority class
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight = {0: weights[0], 1: weights[1]}
print(f"\nClass weights: not_pushup={weights[0]:.2f}, pushup={weights[1]:.2f}")

# Build model — input shape matches our feature count
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Train with early stopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate
y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['NOT_PUSHUP', 'PUSHUP']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("  [[TN  FP]")
print("   [FN  TP]]")

# Plot training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='train')
ax1.plot(history.history['val_loss'], label='val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history.history['accuracy'], label='train')
ax2.plot(history.history['val_accuracy'], label='val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.show()

# Show some predictions on pushup samples
pushup_test_mask = y_test == 1
pushup_probs = y_pred_prob[pushup_test_mask]
print(f"\nPushup sample predictions (should be close to 1.0):")
print(f"  Mean: {pushup_probs.mean():.3f}, Min: {pushup_probs.min():.3f}, Max: {pushup_probs.max():.3f}")
print(f"\nModel size: {model.count_params()} parameters")